# How a transaction gets its category

This notebook follows a few bank transactions through the pipeline, one step per cell, from the raw
export row to the stored row with its **Type**, **Category**, **Sub-category**, **Method** and the
rule that decided them (**filed by**).

Every step calls the dashboard's own code (`dashboard/rules.js` and `dashboard/engine.js`, loaded
into Node by `scripts/load-engine.js`), so what you see here is exactly what the dashboard does when
you upload a file. Nothing is re-implemented in Python.

**Synthetic data only.** The export below is invented. This notebook never reads `data/` or a real
bank export. Clear the outputs before committing anyway.

Needs Python 3 with pandas and Node 18+ on the PATH (Windows, macOS or Linux).

| Step | What happens | Engine function |
|---|---|---|
| 1 | Find the date, description, amount and balance columns | `findColumn` |
| 2 | Read the date and the amount | `parseDateFlexible`, `parseAmount` |
| 3 | Normalize the description | `normalize` |
| 4 | Internal transfer? Then it is a Transfer and stops here | `internalTransferSub` |
| 5 | Income, expense or refund | `isRefund` |
| 6 | Category and sub-category: the rule ladder | `categorizeExplained` |
| 7 | Method: the payment rail | `overlayMethod` |
| 8 | Meal card files | `rawRowsToTransactions(..., {mealCard})` |
| 9 | Transaction id | `makeTransactionId` |
| 10 | All of it in one call, checked against steps 1 to 9 | `rawRowsToTransactions` |
| 11 | Merge into the store, and your review | the dashboard |

## Setup

`js()` runs one JavaScript expression inside the dashboard's engine and returns the result as Python
data. Values passed as keyword arguments become variables in that scope; `rules=` sets your own
rules (`data/category_overrides.json`) for that call.

In [ ]:
import json, subprocess
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 60)

REPO = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
LOADER = (REPO / "scripts" / "load-engine.js").as_posix()   # forward slashes work on Windows too


def js(expression, rules=None, **values):
    """Evaluate `expression` in the dashboard's engine and return the result."""
    program = (
        f"const $ = require({json.dumps(LOADER)}).loadEngine();"
        "const input = JSON.parse(require('fs').readFileSync(0, 'utf8'));"
        "Object.assign($.context, input.values);"
        "$.context.__rules = input.rules; $('overrides = __rules');"
        f"const out = $({json.dumps(expression)});"
        "process.stdout.write(JSON.stringify(out === undefined ? null : out));"
    )
    run = subprocess.run(["node", "-e", program], input=json.dumps({"values": values, "rules": rules or []}),
                         capture_output=True, text=True, encoding="utf-8")
    if run.returncode:
        raise RuntimeError(run.stderr)
    return json.loads(run.stdout)


js("TAXONOMY.Expense.Alimentação")   # quick check that Node and the engine load

## The input: a bank export

The shape the bank gives you (Portuguese column names, day-first dates, amounts as text with a
decimal comma, newest first). Each row is chosen to take a different path through the pipeline.

One of your own rules is also defined here: anything with the keyword "clube exemplo" is
Desporto / Futebol. In the dashboard it would come from ticking **Remember as a rule** in Review.

In [ ]:
export = pd.DataFrame([
    ("08-11-2026", "Compra Continente Lisboa",          "-42,10",   "1.150,00"),
    ("08-11-2026", "Mbway - Cinemas NOS",               "-7,00",    "1.192,10"),
    ("07-11-2026", "Transferência entre contas",        "-200,00",  "1.199,10"),
    ("07-11-2026", "Compra Easypay Loja Online",        "-19,99",   "1.399,10"),
    ("06-11-2026", "Trf. MB WAY para Amigo Exemplo",      "-15,00",   "1.419,09"),
    ("06-11-2026", "Transferência de Colega Exemplo",      "25,00",    "1.434,09"),
    ("05-11-2026", "Devolução Zara",                    "29,99",    "1.409,09"),
    ("05-11-2026", "Clube Exemplo Quota Novembro",      "-30,00",   "1.379,10"),
    ("04-11-2026", "Loja Desconhecida 123",             "-5,00",    "1.409,10"),
    ("03-11-2026", "Cafe Central",                      "-1,20",    "1.414,10"),
    ("03-11-2026", "Cafe Central",                      "-1,20",    "1.415,30"),
    ("02-11-2026", "Ordenado Empresa Exemplo",          "1.234,56", "1.416,50"),
], columns=["Data Operação", "Descrição", "Montante( EUR )", "Saldo Contabilístico( EUR )"])

MY_RULES = [{"keywords": ["clube exemplo"], "type": "expense",
             "category": "Desporto", "sub_category": "Futebol", "method": ""}]

raw_rows = export.to_dict("records")
export

## Step 1: find the columns

Banks name their columns differently, so each field is found through a list of accepted names
(`DATE_ALIASES`, `DESCRIPTION_ALIASES`, `AMOUNT_ALIASES`, `BALANCE_ALIASES` in engine.js). Without a
date, a description and an amount the file is refused; the balance is optional.

In [ ]:
columns = list(export.columns)
pd.Series(js("""({
    date: findColumn(cols, DATE_ALIASES),
    description: findColumn(cols, DESCRIPTION_ALIASES),
    amount: findColumn(cols, AMOUNT_ALIASES),
    balance: findColumn(cols, BALANCE_ALIASES),
})""", cols=columns), name="column found")

## Step 2: read the date and the amount

Dates are day-first (`08-11-2026`) or ISO. Amounts are read in either convention: whichever of `.` or
`,` comes last is the decimal point, so `1.234,56` is 1234.56. A row whose date or amount can't be
read is skipped and counted, never guessed.

In [ ]:
parsed = js("""rows.map(r => ({
    date: parseDateFlexible(r["Data Operação"]),
    amount: parseAmount(r["Montante( EUR )"]),
}))""", rows=raw_rows)

step2 = export[["Data Operação", "Montante( EUR )", "Descrição"]].copy()
step2["Date"] = [p["date"] for p in parsed]
step2["amount"] = [p["amount"] for p in parsed]
step2

## Step 3: normalize the description

Lowercase, accents stripped, spaces collapsed. Every rule is written against this form, so
"Transferência" and "transferencia" match the same pattern. The original description is still what
gets stored (in **Notes**); the normalized form is only used for matching.

In [ ]:
step3 = pd.DataFrame({"Descrição": export["Descrição"],
                      "normalized": js("descs.map(normalize)", descs=list(export["Descrição"]))})
step3

## Step 4: internal transfer?

Money moving between your own accounts is neither income nor spending. If the description matches
`INTERNAL_RULES` in rules.js, the row becomes **Type Transfer**, Category *Transferências Internas*,
keeps its signed amount (+ in, - out) and skips steps 5 to 7. It is left out of every income and
spending total.

In [ ]:
step4 = pd.DataFrame({"Descrição": export["Descrição"],
                      "internal transfer": js("descs.map(internalTransferSub)", descs=list(export["Descrição"]))})
step4["internal transfer"] = step4["internal transfer"].fillna("no")
step4

## Step 5: income, expense or refund

The sign decides first: money out is an **Expense**, money in is **Income**. The exception is a refund
("Devolução", "Estorno"): money in that undoes a purchase. It is stored as an Expense with a
**negative** amount under the purchase's category, so it nets against that spending and never
inflates income.

In [ ]:
amounts = [p["amount"] for p in parsed]
refund = js("descs.map((d, i) => isRefund(d, amounts[i]))", descs=list(export["Descrição"]), amounts=amounts)

step5 = pd.DataFrame({"Descrição": export["Descrição"], "amount": amounts, "refund": refund})
step5["Type"] = ["Transfer" if t != "no" else ("Expense" if r or a < 0 else "Income")
                 for t, r, a in zip(step4["internal transfer"], refund, amounts)]
step5["stored amount"] = [a if t == "Transfer" else (-abs(a) if r else abs(a))
                          for t, r, a in zip(step5["Type"], refund, amounts)]
step5

## Step 6: category and sub-category, the rule ladder

The engine tries each rung from the top and stops at the first that matches:

1. **Payment gateway** (Easypay, Stripe, ...): the real merchant is hidden, so the row goes to
   *Outros / Por Classificar* for you to review.
2. **Your rules** (`data/category_overrides.json`).
3. **Built-in rules** (`PURPOSE_INCOME_RULES` / `PURPOSE_EXPENSE_RULES` in rules.js), in order.
4. **A transfer to or from a person** (MBWay or bank transfer with no other match):
   *Transferências Pessoais*.
5. **Nothing matched**: *Outros / Por Classificar*, an honest "I don't know".

A row you reviewed by hand sits above all of these and is never re-derived (step 11).

The table shows, for each row, which rungs would match and which one won. Internal transfers are
left out: step 4 already settled them.

In [ ]:
LADDER = """descs.map((d, i) => {
    const n = normalize(d), amount = amounts[i];
    const income = amount > 0 && !REFUND_RE.test(n);
    const builtIn = (income ? PURPOSE_INCOME_RULES : PURPOSE_EXPENSE_RULES)
        .find(([patterns]) => compileRule(patterns).test(n));
    const result = categorizeExplained(d, amount);
    return {
        gateway: GATEWAY_RE.test(n),
        "your rule": !!matchOverride(n, income ? "income" : "expense"),
        "built-in": builtIn ? builtIn[1] + " / " + builtIn[2] : "",
        Category: result.category,
        "Sub-category": result.sub,
        "filed by": result.source,
    };
})"""

not_transfer = step5["Type"] != "Transfer"
ladder = pd.DataFrame(js(LADDER, rules=MY_RULES,
                         descs=list(export["Descrição"][not_transfer]),
                         amounts=[a for a, keep in zip(amounts, not_transfer) if keep]))
ladder.insert(0, "Descrição", list(export["Descrição"][not_transfer]))
ladder

Things to notice:

- **Compra Easypay Loja Online** names a gateway, so it waits for your review even though "loja
  online" could have matched something lower down.
- **Clube Exemplo Quota** matched your rule, so the built-in rules were never asked.
- **Devolução Zara** is categorized on the expense side (Lazer / Roupa) because it is a refund.
- **Trf. MB WAY para Amigo Exemplo** and **Transferência de Colega Exemplo** matched no purpose rule, so
  they fall to the person-transfer rung: money out waits in *Outros / Transferências Pessoais*,
  money in is income under *Transferências Pessoais*.
- **Loja Desconhecida 123** matched nothing and is honestly left as *Por Classificar*.
- **filed by** names a built-in rule by its first three patterns only. "Cafe Central" matched on
  "cafe", further along the same rule (`PURPOSE_EXPENSE_RULES` in rules.js).

## Step 7: method, the payment rail

The method answers "how did the money move", separately from "what was it for". A rule suggests a
default (Cartão for most shops, Débito Direto for bills), but if the description names a rail it
wins: "Mbway - Cinemas NOS" is Lazer / Cinema **and** MBWay.

The meal card (Cartão Alimentação) is never read from a description; see step 8.

In [ ]:
METHOD = """descs.map((d, i) => {
    const r = categorizeExplained(d, amounts[i]);
    const n = normalize(d);
    const income = amounts[i] > 0 && !REFUND_RE.test(n);
    const builtIn = (income ? PURPOSE_INCOME_RULES : PURPOSE_EXPENSE_RULES)
        .find(([patterns]) => compileRule(patterns).test(n));
    return {
        "rule suggests": builtIn ? builtIn[3] : (income ? "Transferência" : "Cartão") + " (default)",
        "MBWay named": RAIL_MBWAY.test(n),
        "transfer named": RAIL_TRANSFER.test(n),
        Method: r.method,
    };
})"""

step7 = pd.DataFrame(js(METHOD, rules=MY_RULES,
                        descs=list(export["Descrição"][not_transfer]),
                        amounts=[a for a, keep in zip(amounts, not_transfer) if keep]))
step7.insert(0, "Descrição", list(export["Descrição"][not_transfer]))
step7

## Step 8: meal card files

A meal card export (file name with "alim", "refei", "meal", "coverflex" or "edenred", or the
"meal card exports" box ticked at upload) changes two things: every row gets Method
**Cartão Alimentação**, and money in is the meal allowance, **Salário Alimentação**. This is the
only way a row gets that method, which keeps the "exclude meal card" toggle trustworthy.

In [ ]:
meal_rows = [
    {"Data": "02-11-2026", "Descrição": "Carregamento Mensal", "Montante": "180,00"},
    {"Data": "05-11-2026", "Descrição": "Compra Continente Lisboa", "Montante": "-23,40"},
]
print("is a meal card file name:", js("isMealCardFile(name)", name="MealCard_Nov.xlsx"))
pd.DataFrame(js("rawRowsToTransactions(rows, 'MealCard_Nov.xlsx', { mealCard: true }).rows",
                rows=meal_rows))[["Notes", "Type", "Category", "Sub-category", "Method", "Amount (€)"]]

## Step 9: transaction id

`sha1(date | lowercased description | amount)`, first 16 hex characters. Because it is derived from
the transaction itself, re-importing a file, or two exports that overlap, add nothing new.

Two genuinely identical transactions in one file (two coffees, same place, same day, same amount)
would hash the same, so the second gets an occurrence number: `|2`. The first keeps the plain key,
so ids created before this existed did not change.

In [ ]:
coffees = js("""[1, 2].map(k => ({
    occurrence: k,
    key: '2026-11-03|cafe central|-1.20' + (k > 1 ? '|' + k : ''),
    transaction_id: makeTransactionId('2026-11-03', 'Cafe Central', -1.2, k),
}))""")
pd.DataFrame(coffees)

## Step 10: the whole pipeline in one call

`rawRowsToTransactions()` is what the dashboard calls on upload: steps 1 to 9 in one go. The check
below confirms it agrees with the step-by-step tables above, so the walkthrough really describes the
code.

In [ ]:
result = js("rawRowsToTransactions(rows, 'Bank_Nov.xls')", rules=MY_RULES, rows=raw_rows)
store_rows = pd.DataFrame(result["rows"])
print("rows read:", len(store_rows), "| rows skipped:", result["skipped"])

assert list(store_rows["Type"]) == list(step5["Type"])
assert list(store_rows["Amount (€)"]) == [round(a, 2) for a in step5["stored amount"]]
assert list(store_rows.loc[not_transfer.values, "Category"]) == list(ladder["Category"])
assert list(store_rows.loc[not_transfer.values, "Sub-category"]) == list(ladder["Sub-category"])
assert list(store_rows.loc[not_transfer.values, "Method"]) == list(step7["Method"])
assert set(store_rows["transaction_id"]) >= {c["transaction_id"] for c in coffees}
print("matches steps 1 to 9")

store_rows[["Date", "Notes", "Type", "Category", "Sub-category", "Method", "Amount (€)", "transaction_id"]]

## Step 11: merge into the store, and your review

What happens next lives in the dashboard (`index.html`) and the local server:

- **Merge.** Rows whose `transaction_id` is already in `data/transactions.csv` are duplicates and are
  ignored; the rest are appended. The import report lists added, duplicate and skipped rows per
  file. The server refuses any save that would remove a transaction already in the store.
- **Needs review.** Rows filed by the gateway, person-transfer or fallback rungs are what the Review
  tab asks you about.
- **Your review wins.** When you file a row in the review panel it gets a `Reviewed At` stamp and is
  never re-categorized again, by any rule. Tick **Remember as a rule** and the keyword joins your
  rules (rung 2), so the next import files it on its own.

Re-importing the same file adds nothing:

In [ ]:
again = js("rawRowsToTransactions(rows, 'Bank_Nov.xls').rows.map(r => r.transaction_id)", rules=MY_RULES, rows=raw_rows)
new_ids = set(again) - set(store_rows["transaction_id"])
print(f"re-import: {len(again)} rows read, {len(new_ids)} new")

## Summary: where each transaction ended up

In [ ]:
summary = store_rows[["Notes", "Type", "Category", "Sub-category", "Method", "Amount (€)"]].copy()
filed = dict(zip(ladder["Descrição"], ladder["filed by"]))
summary["filed by"] = [filed.get(n, "internal transfer") for n in summary["Notes"]]
summary